In [630]:
import pandas as pd
import numpy as np

from pathlib import Path

In [631]:
exam_result_path = Path("/Users/abelluc/FAUbox/DigiKolleg/ADS")

In [632]:
exam_results = pd.read_csv(exam_result_path.joinpath("ADS_Exam_2023_results.csv"), sep=";")
exam_results.head()

In [633]:
exam_results.set_index("Login", inplace=True)
exam_results.index.names = ["student_id"]
exam_results.head()

In [634]:
points = exam_results["Test Results in Points"]
points.hist(bins=100)

In [635]:
points = pd.DataFrame(points)
points.columns = ["points"]
points.head()

In [636]:
pass_score = 45.0
max_score = 100.0
bin_size = round((max_score-pass_score) / 11, 1)
bin_size

In [637]:
grades = [1.0, 1.33, 1.66, 2.0, 2.33, 2.66, 3.0, 3.33, 3.66, 4.0, 4.33, 5.0]
grades = grades[::-1]

In [638]:
min_points = [pass_score + bin_size * i for i in range(-2,len(grades)-1)]
min_points

In [639]:
grade_map = pd.DataFrame(zip(min_points, grades), columns=["min_points", "grade"])
grade_map["max_points"] = grade_map["min_points"] + bin_size - 0.1
grade_map.set_index("grade", inplace=True)
grade_map.loc[1.0, "max_points"] = max_score
grade_map.loc[5.0, "min_points"] = 0.0

grade_map.sort_index(inplace=True)

grade_map

In [640]:
# return the value of the index of grade_map where points is between min_points and max_points
points["grade"] = [grade_map[(grade_map["min_points"] <= point) & (grade_map["max_points"] >= point)].index[0] for point in points["points"]]
points


In [641]:
# Without Bonus

points["grade"].value_counts().sort_index().plot(kind="bar")

In [642]:
bonus_points = pd.read_csv("/Users/abelluc/Code/ads_exercise/total_score/2023_total.csv")
bonus_points.set_index("identifier", inplace=True)
bonus_points.index.names = ["student_id"]
bonus_points

In [643]:
# join points and bonus_points

points = points.join(bonus_points)
points.drop("student_id", axis=1, inplace=True)
points

In [644]:
# replace values in score_percent with 0.33, 0.66, 1.0
points["bonus"] = np.zeros(len(points))

points["bonus"][points["score_percent"] >= 95] = 0.66
points["score_percent"][points["score_percent"] >= 95] = 0

points["bonus"][points["score_percent"] >= 90] = 0.33

# replace nan with 0
points.fillna(0, inplace=True)

points.drop("score_percent", axis=1, inplace=True)
points

In [645]:
points["grade_with_bonus"] = points["grade"] - points["bonus"]

# set grade_with_bonus to 1.0 if grade_with_bonus is smaller than 1.0
points["grade_with_bonus"][points["grade_with_bonus"] < 1.0] = 1.0

# set grade_with_bonus to grade value if grade is greater than 4.0
points["grade_with_bonus"][points["grade"] > 4.0] = points["grade"]

points

In [648]:
points = round(points, 1)
points.sort_values("grade", inplace=True)
points

In [649]:
points.describe()

In [650]:
points["grade_with_bonus"].value_counts().sort_index().plot(kind="bar")

In [651]:
# save to csv
points.to_csv(exam_result_path.joinpath("ADS_Exam_2023_results_final.csv"))